In [1]:
import pickle

In [2]:
def load_obj( name ):
    """
    Load dataset from pickle file.
    :param name: Full pathname of the pickle file
    :return: Dataset type of dictionary
    """
    with open( name , 'rb') as f:
        return pickle.load(f)

In [ ]:
dataset = load_obj('baseline_pc50_STRING.pkl') 
    
#nlabels = 2
    #networks_name = ['string','CPDB','pcnet','pcnet','pcnet','pcnet','string']
        

#data = dataset[0]
#data.adj_t = data.adj_t.to_symmetric()
        

#x = data.x


In [ ]:
import torch
from torch_geometric.utils import to_networkx, from_networkx
import torch_geometric.transforms as T
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

data = Data(x=dataset['feature'], y=dataset['label'], edge_index=dataset['edge_index'], mask=mask, node_names=dataset['node_name'])

In [10]:
import pandas as pd
import numpy as np
import networkx as nx
import h5py, os, sys
import random
import matplotlib.pyplot as plt
import seaborn as sns

swapping_percentages = [0, 0.25, 0.5, 0.75, 1]

In [ ]:
G = to_networkx(data, to_undirected=False)

In [ ]:
import random

edges = list(G.edges())

disturb_ratios = [0.25, 0.5, 0.75, 1]

rewired_networks = [G]

for ratio in disturb_ratios:

    swap_count = int(len(edges) * ratio)
    

    G_copy = G.copy()
    
    edges_to_swap = random.sample(edges, swap_count)
    
    G_copy.remove_edges_from(edges_to_swap)
    
    while len(G_copy.edges()) < len(edges) - len(edges_to_swap) + swap_count:
        u, v = random.sample(G_copy.nodes(), 2)
        if not G_copy.has_edge(u, v):  
            G_copy.add_edge(u, v)
    
    rewired_networks.append(G_copy)



In [14]:
perturbed_networks = rewired_networks

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

def plot_degree_distribution(G, filename, ratio):
    degrees = [d for n, d in G.degree()]
    
    plt.figure(figsize=(4, 4))
    plt.hist(degrees, bins=range(1, max(degrees) + 2), color="#69b3a2", edgecolor="black", alpha=0.7)
    plt.xlabel("Degree", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.title(f"Perturbed Fraction: {ratio}", fontsize=16)
    plt.xticks(
        ticks=range(0, max(degrees) + 1, max(1, max(degrees) // 10)),
        fontsize=12,
        rotation=45
    )
    plt.yticks(fontsize=12)
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(f"{filename}.pdf")
    plt.close()

plot_degree_distribution(perturbed_networks[0], "degree_dist_0",'0')
plot_degree_distribution(perturbed_networks[1], "degree_dist_0.25",'0.25')
plot_degree_distribution(perturbed_networks[2], "degree_dist_0.5",'0.5')
plot_degree_distribution(perturbed_networks[3], "degree_dist_0.75",'0.75')
plot_degree_distribution(perturbed_networks[4], "degree_dist_1",'1')


In [59]:
for i in range(len(perturbed_networks)):
    output_file = "string_network_perturbed_{}.txt".format(swapping_percentages[i])  # 输出文件名.txt"
    with open(output_file, "w") as f:
        for u, v in perturbed_networks[i].edges():
            f.write(f"{u} {v}\n")

In [ ]:
#dataset['node_name']

output_file = "string_edge_index_gname_mapping.txt"

with open(output_file, "w") as f:
    for idx, line in enumerate(dataset['node_name']):
        f.write(f"{idx} {line}\n")

In [ ]:
swapping_percentages = [0, 0.25, 0.5, 0.75, 1]
perturbation_folder = './perturbations_10_5CV/'

In [ ]:
data = from_networkx(perturbed_networks[0])
edge_index = data.edge_index
edge_index

In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
import networkx as nx
import os

from torch_geometric.utils import from_networkx


for i in range(len(perturbed_networks)):
    print(perturbed_networks[i].number_of_nodes()) 
    data = from_networkx(perturbed_networks[i])
    edge_index = data.edge_index

    new_dataset = dict()
    new_dataset['edge_index'] = edge_index
    new_dataset['label'] = dataset['label']
    new_dataset['node_name'] = dataset['node_name']
    new_dataset['feature'] = dataset['feature']
    new_dataset['split_set'] = dataset['split_set']


    fname = os.path.join(perturbation_folder, 'cancer_full_expression_pc50_10_5CV_string_{}_network_perturbation.pkl'.format(str(swapping_percentages[i]).replace('.', '_')))
    with open(fname, 'wb') as f:
        pickle.dump(new_dataset, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
def _swap_row(matrix, idx_from, idx_to):
    tmp = matrix[idx_from]
    matrix[idx_from] = matrix[idx_to]
    matrix[idx_to] = tmp
"""
def perturb_features(features, percentage, max_tries=100):
    n_swaps = percentage * features.shape[0]
    print (n_swaps)
    available_nodes = list(np.arange(features.shape[0]))
    for swap in range(int(n_swaps)):
        found_new_swap = False
        num_tries = 0
        while not found_new_swap or num_tries < max_tries:
            fro, to = np.random.randint(features.shape[0], size=2)
            if not fro in already_perturbed and not to in already_perturbed:
                found_new_swap = True
                _swap_row(features, fro, to)
                already_perturbed.append(fro)
                already_perturbed.append(to)
            else:
                num_tries += 1
    print (len(already_perturbed))
    return features, already_perturbed
"""

def perturb_features(features, percentage, max_tries=100):
    n_swaps = percentage * features.shape[0]
    available_nodes = list(np.arange(features.shape[0]))
    for swap in range(int(n_swaps)):
        fro = random.choice(available_nodes)
        available_nodes.remove(fro)
        to = random.choice(np.arange(features.shape[0]))
        _swap_row(features, fro, to)
    return features

features = dataset['feature'].numpy()
perturbed_features = [features]
for percentages in swapping_percentages[1:]:
    if type(percentages) == float or type(percentages) == int:
        perturbed_features.append(perturb_features(features.copy(), percentages))
        print ("Swapped {}% of the features".format(percentages*100))
    else:
        perturbed_features.append(np.random.randn(features.shape[0], 1))
        #perturbed_features.append(np.random.randn(*features.shape))
        print ("Created random features")

In [ ]:
for i in range(len(perturbed_features)):

    new_dataset = dict()
    new_dataset['edge_index'] = dataset['edge_index']
    new_dataset['label'] = dataset['label']
    new_dataset['node_name'] = dataset['node_name']
    new_dataset['feature'] = torch.tensor(perturbed_features[i])
    new_dataset['split_set'] = dataset['split_set']


    fname = os.path.join(perturbation_folder, 'cancer_full_expression_pc50_10_5CV_string_{}_feature_perturbation.pkl'.format(str(swapping_percentages[i]).replace('.', '_')))
    with open(fname, 'wb') as f:
        pickle.dump(new_dataset, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
for i in range(len(perturbed_features)):

    data = from_networkx(perturbed_networks[i])
    edge_index = data.edge_index

    new_dataset = dict()
    new_dataset['edge_index'] = edge_index
    new_dataset['label'] = dataset['label']
    new_dataset['node_name'] = dataset['node_name']
    new_dataset['feature'] = torch.tensor(perturbed_features[i])
    new_dataset['split_set'] = dataset['split_set']


    fname = os.path.join(perturbation_folder, 'cancer_full_expression_pc50_10_5CV_string_{}_networkfeature_perturbation.pkl'.format(str(swapping_percentages[i]).replace('.', '_')))
    with open(fname, 'wb') as f:
        pickle.dump(new_dataset, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
for i in range(len(perturbed_networks)):
    print(perturbed_networks[i].number_of_nodes()) 
    data_ = from_networkx(perturbed_networks[i])
    edge_index = data_.edge_index

    pyg_graphs = []

    #row, col, edge_attr = data.adj_t.t().coo()
    #edge_index = torch.stack([row, col], dim=0)

    weights = torch.FloatTensor([1.0 for G in range(len(edge_index[0]))])
    
    pyg_graph = Data(x = data.x,  edge_index = edge_index, edge_attr= weights, y = data.y)

    pyg_graph.train_mask = data.train_mask
    pyg_graph.valid_mask = data.valid_mask
    pyg_graph.test_mask = data.test_mask
    

    pyg_graph = T.ToSparseTensor(remove_edge_index=True)(pyg_graph)
    pyg_graph.k_sets_net = data.k_sets_net

    pyg_graphs.append(pyg_graph)

    fname = os.path.join(perturbation_folder, 'subgraph_essential_task_subgraph_STRING_{}_perturbation.pkl'.format(str(swapping_percentages[i]).replace('.', '_')))
    with open(fname, 'wb') as f:
        pickle.dump(pyg_graphs, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
for i in range(len(perturbed_networks)):
    pyg_graphs = []

    row, col, edge_attr = data.adj_t.t().coo()
    edge_index = torch.stack([row, col], dim=0)

    #weights = torch.FloatTensor([1.0 for G in range(len(edge_index[0]))])
    
    pyg_graph = Data(x = torch.tensor(perturbed_features[i], dtype=torch.float),  edge_index = edge_index, edge_attr= data.edge_attr, y = data.y)

    pyg_graph.train_mask = data.train_mask
    pyg_graph.valid_mask = data.valid_mask
    pyg_graph.test_mask = data.test_mask

    pyg_graph = T.ToSparseTensor(remove_edge_index=True)(pyg_graph)
    pyg_graph.k_sets_net = data.k_sets_net



    pyg_graphs.append(pyg_graph)

    fname = os.path.join(perturbation_folder, 'subgraph_essential_task_subgraph_STRING_{}_feature_perturbation.pkl'.format(str(swapping_percentages[i]).replace('.', '_')))
    with open(fname, 'wb') as f:
        pickle.dump(pyg_graphs, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
for i in range(len(perturbed_networks)):
    pyg_graphs = []


    data_ = from_networkx(perturbed_networks[i])
    edge_index = data_.edge_index

    weights = torch.FloatTensor([1.0 for G in range(len(edge_index[0]))])
    
    pyg_graph = Data(x = torch.tensor(perturbed_features[i], dtype=torch.float),  edge_index = edge_index, edge_attr= weights, y = data.y)

    pyg_graph.train_mask = data.train_mask
    pyg_graph.valid_mask = data.valid_mask
    pyg_graph.test_mask = data.test_mask

    pyg_graph = T.ToSparseTensor(remove_edge_index=True)(pyg_graph)
    pyg_graph.k_sets_net = data.k_sets_net


    pyg_graphs.append(pyg_graph)

    fname = os.path.join(perturbation_folder, 'subgraph_essential_task_subgraph_STRING_{}_networkfeature_perturbation.pkl'.format(str(swapping_percentages[i]).replace('.', '_')))
    with open(fname, 'wb') as f:
        pickle.dump(pyg_graphs, f, pickle.HIGHEST_PROTOCOL)